## TypeScript Translation Notes
* `RecursiveSet` provides mathematical sets where equality is determined by content.
* Using `filterMap` computes mapped values in a single pass without intermediate allocation, serving as an efficient alternative to Python's set comprehensions.
* `Tuple` serves as an immutable, hashable sequence of values. Destructuring is favored over the `get` method to access values.
* `RecursiveMap` acts as our dictionary replacement, natively supporting complex `Tuple` object keys.
* The `.some()` method is utilized for its fail-fast behavior ($O(1)$ best case) to cleanly break out of search loops early.

In [1]:
import { RecursiveSet, RecursiveMap, Tuple } from "recursive-set";

In [2]:
function range(n: number): RecursiveSet<number> {
    const result = new RecursiveSet<number>();
    for (let i = 0; i < n; i++) {
        result.add(i);
    }
    return result;
}

# Missionaries and Infidels

We illustrate the notion of a search problem with the following example, which is also known as the
<a href="https://en.wikipedia.org/wiki/Missionaries_and_cannibals_problem">missionaries and cannibals problem</a>:
Three *missionaries* and three *infidels* have to cross a river that runs from the north to the south.
Initially, both the missionaries and the infidels are on the western shore.  There is just one small boat and
that boat can carry at most two passengers.  Both the missionaries and the infidels can steer the boat.
However, if at any time the missionaries are confronted with a majority of infidels on either shore of the
river, then the missionaries have a problem. Below is an artist's rendition of the problem.

![Missionaries and Infidels](missionaries-and-infidels.png)



$\texttt{no_problem}(m, i)$ is true if there is no problem on either side.
$m$ and $i$ are the number of missionaries and infidels on the left shore.
Hence there are $3-m$ missionaries and $3-i$ infidels on the right shore.

There is no problem if either all missionaries are on the same side of the river or 
if the number of missionaries is the same as the number of infidels.

In [3]:
function no_problem(M: number, I: number): boolean { 
    return M === 0 || M === 3 || M === I;
}

A state is represented as a `Tuple`.  The tuple $(m, i, b)$ specifies that there are
  - $m$ missionaries,
  - $i$ infidels, and
  - $b$ boats

on the *western* shore of the river.  This implies that there are 
$3 - m$ missionaries, $3 - i$ infidels, and $1 - b$ boats on the *eastern* shore.

The type of a state is defined as a `Tuple` of three integers.

In [4]:
type State = Tuple<[number, number, number]>;

The function `next_states` takes a given `state` and computes the set of states that can be reached from `state` by crossing the river. 

In [ ]:
function next_states(state: State): RecursiveSet<State> {
    const [m, i, b] = state;
    if (b === 1) {
        return range(m + 1).cartesianProduct(range(i + 1)).filterMap(
            (t) => {
                const [mb, ib] = t;
                return 1 <= mb + ib && mb + ib <= 2 && no_problem(m - mb, i - ib);
            },
            (t) => {
                const [mb, ib] = t;
                return new Tuple(m - mb, i - ib, 0);
            }
        );
    } else {
        return range(3 - m + 1).cartesianProduct(range(3 - i + 1)).filterMap(
            (t) => {
                const [mb, ib] = t;
                return 1 <= mb + ib && mb + ib <= 2 && no_problem(m + mb, i + ib);
            },
            (t) => {
                const [mb, ib] = t;
                return new Tuple(m + mb, i + ib, 1);
            }
        );
    }
}

Initially, all missionaries, all infidels and the boat are on the left shore.
The goal is to have everybody on the right shore, hence the numbers on the left shore
should all be $0$.

In [ ]:
const start = new Tuple(3, 3, 1);
const goal  = new Tuple(0, 0, 0);

In [ ]:
console.log(next_states(start).toString());

# Breadth First Search



In [ ]:
type NxtStFct = (state: State) => RecursiveSet<State>;

Given a `state` and a parent map `Parent`, the function `path_to` returns a path leading from `start` to the given `state`.

In [ ]:
function path_to(state: State, Parent: RecursiveMap<State, State>): State[] {
    const p = Parent.get(state);
    if (!p) throw new Error("State not found in Parent map");
    if (p.equals(state)) {
        return [state];
    }
    return [...path_to(p, Parent), state];
}

---

The function `search` takes three arguments to solve a *search problem*:
- `start` is the *start state* of the search problem,
- `goal` is the *goal state*, and
- `next_states` is a function with signature $\texttt{next_states}:Q \rightarrow 2^Q$, where $Q$ is the set of states.
   For every state $s \in Q$, $\texttt{next_states}(s)$ is the set of states that can be reached from $s$ in one step.

If successful, `search` returns a path from `start` to `goal` that is a solution of the search problem
$$ \langle Q, \texttt{next_states}, \texttt{start}, \texttt{goal} \rangle. $$
The implementation of `search` uses the algorithm *breadth first search* to find a path from `start` to `goal`.

At the start of the $n^\textrm{th}$ iteration of the `while` loop, the following invariants are satisfied:
* `Frontier` contains exactly those states that have distance of $n-1$ from `start`.
* `Visited`  contains those states that have distance from start that is less than `n-1`.
* `Parent` is a recursive map. The keys of this map are all states from the sets `Visited`, `Frontier`, and `NewFrontier`.    Furthermore, the following invariant holds for all states $x \not= \texttt{start}$:
  - If $x = \texttt{Parent}[y]$, then $y \in \texttt{next_states}(x)$.  

In [ ]:
function search(start: State, goal: State, next_states: NxtStFct): State[] | null {
    let Frontier = new RecursiveSet<State>(start);
    let Visited = new RecursiveSet<State>();
    const Parent = new RecursiveMap<State, State>();
    Parent.set(start, start);

    while (Frontier.size > 0) {
        const NewFrontier = new RecursiveSet<State>();
        let foundGoal = false;

        Frontier.some((s) => {
            return next_states(s).some((ns) => {
                if (!Visited.has(ns)) {
                    NewFrontier.add(ns);
                    Parent.set(ns, s);
                    if (ns.equals(goal)) {
                        foundGoal = true;
                        return true; // Break the inner loop early
                    }
                }
                return false;
            });
        });

        if (foundGoal) {
            console.log("number of states: ", Visited.size + Frontier.size + NewFrontier.size);
            return path_to(goal, Parent);
        }

        Visited = Visited.union(Frontier);
        Frontier = NewFrontier;
        console.log(Frontier.size);
    }
    return null;
}

In [ ]:
const path = search(start, goal, next_states);
if (path) {
    console.log(path.map(p => p.toString()).join("\n"));
}